In [4]:
import argparse
import os

import torch
from torch import nn, optim

from dataloader import PoetryDataset, PoetryVocabulary, create_dataloader, load_qiyan_jueju_corpus
from models import PoetryGenerator
from utils import generate_poems, plot_training_curve, set_seed, train_model

# Configuration parameters
data_root = "./datasets"
save_dir = "./results"
batch_size = 64
epochs = 150
dropout = 0.2
seed = 42
model_type = "lstm"  # choices: ["lstm", "rnn"]
generate_count = 3

set_seed(seed)

poems = load_qiyan_jueju_corpus(data_root)
vocabulary = PoetryVocabulary(poems, min_freq=2)
dataset = PoetryDataset(poems, vocabulary)
train_loader = create_dataloader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

try:
    torch.zeros(1, device="cuda")
    device = torch.device("cuda")
except Exception as error:
    print(f"CUDA is unavailable at runtime, fallback to CPU: {error}")
    device = torch.device("cpu")

model = PoetryGenerator(
    vocab_size=len(vocabulary),
    embedding_dim=256,
    hidden_dim=512,
    num_layers=2,
    dropout=dropout,
    model_type=model_type,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

os.makedirs(save_dir, exist_ok=True)
checkpoint_path = os.path.join(save_dir, "best_model.pth")
curve_path = os.path.join(save_dir, "training_loss_curve.png")

print(f"Corpus size: {len(poems)}")
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Model type: {model_type.upper()}")

history = train_model(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=epochs,
    save_path=checkpoint_path,
    history_path=None,
)

plot_training_curve(history["loss"], curve_path)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint)

poems = generate_poems(
    model=model,
    vocabulary=vocabulary,
    device=device,
    prefix="\u660e\u6708",
    count=generate_count,
    temperature=0.8,
    top_k=8,
)

print("Generated poems:")
for index, poem in enumerate(poems, start=1):
    print(f"Poem {index}")
    print(poem)

Corpus size: 906
Vocabulary size: 1951
Model type: LSTM
Device: cuda
Train samples: 906
Train steps per epoch: 15


Generated poems:
Poem 1
明月無前前水香，一堂唯有一邊塵。
人間訟嘆人難數，到輩春風不自非。
Poem 2
明月無時妙氣陰，圍山遠坐到清深。
吹君舊復桑田變，應用人時萬戶門。
Poem 3
明月新鵝色澤均，此開元得作仙飛。
古今不爲桑田變，知是人間幾歲人。
